# 漫剧投放数据质量审计

## TL;DR

- 2026 年 7 月数据包含 551 行、551 个匿名账号、8 个匿名品牌和 22 个源字段。
- 主键重复、缺失、负值、点击大于展示、播放大于展示均为 0。
- 零展示 17 个、零点击 83 个、零消耗 17 个、零播放 82 个；比率采用安全除法，不删除这些账号。
- 源“播放成本”字段实际等于播放量÷消耗；最终分析改用消耗÷播放量的重算字段。

## Context & Methods

数据粒度是“账号 × 月”，只做单月横截面诊断。检查结构完整性、唯一性、数值边界、零分母和源指标公式。所有金额经过同比例缩放，绝对值只用于本数据集内比较。

## Data

唯一输入是 `data/processed/manga_ad_account_2026_07_anonymized.xlsx` 的 `脱敏数据` 工作表。

In [1]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from manga_ad_analysis.io import load_source
from manga_ad_analysis.metrics import add_account_metrics
from manga_ad_analysis.quality import quality_summary

source_path = PROJECT_ROOT / "data/processed/manga_ad_account_2026_07_anonymized.xlsx"
frame = load_source(source_path)
enriched = add_account_metrics(frame)
print(f"rows={len(frame)}, accounts={frame.account_id.nunique()}, brands={frame.brand_name.nunique()}, columns={frame.shape[1]}")

rows=551, accounts=551, brands=8, columns=22


## Results

### 1. 结构与业务规则检查

In [2]:
audit = quality_summary(frame)
print(audit.to_string(index=False))

                                   check_name  value status severity                          description
                                    row_count    551     通过       信息                                 数据行数
                                 column_count     22     通过       信息                                规范字段数
                                account_count    551     通过       信息                                唯一账号数
                                  brand_count      8     通过       信息                                  品牌数
                                missing_cells      0     通过       正常                               缺失单元格数
                               duplicate_rows      0     通过       正常                               完全重复行数
                 duplicate_account_month_keys      0     通过       正常                         账号—月份复合键重复行数
                     zero_impression_accounts     17     注意       提示                展示量为 0 的账号数；对应比率按空值处理
                          zero_click_accounts 

### 2. 源指标与重算指标复核

In [3]:
valid = enriched[enriched["plays"] > 0].copy()
mismatch_count = (abs(valid["play_cost"] - valid["play_cost_recalc"]) > 1e-6).sum()
inverse_matches = (
    abs(
        enriched.loc[enriched["platform_spend"] > 0, "play_cost"]
        - enriched.loc[enriched["platform_spend"] > 0, "plays"]
        / enriched.loc[enriched["platform_spend"] > 0, "platform_spend"]
    ) <= 1e-6
).sum()
print(f"播放成本公式不一致账号（plays>0）：{mismatch_count}")
print(f"源字段匹配 plays/spend 的账号（spend>0）：{inverse_matches}")
print(enriched[["account_id", "platform_spend", "plays", "play_cost", "play_cost_recalc"]].head(8).to_string(index=False))

播放成本公式不一致账号（plays>0）：469
源字段匹配 plays/spend 的账号（spend>0）：534
account_id  platform_spend  plays  play_cost  play_cost_recalc
   MJ-0482     11382.86340      0   0.000000               NaN
   MJ-0250      9338.76402 210168  22.504905          0.044435
   MJ-0128      5128.31276 119959  23.391514          0.042751
   MJ-0310      4083.21579      0   0.000000               NaN
   MJ-0113      4053.70836  91345  22.533688          0.044378
   MJ-0321      4046.29215      0   0.000000               NaN
   MJ-0204      3749.73149  84538  22.545081          0.044356
   MJ-0304      3520.89819  89883  25.528429          0.039172


## Takeaways

1. 数据结构完整，可以进入汇总与分层；零分母需要保留并显式标记。
2. 品牌 KPI 必须从基础量加权重算，不能直接平均账号比率。
3. 源“播放成本”字段名与实际公式冲突，因此下游统一使用 `play_cost_recalc`。
4. 本次审计不能补齐日/周趋势、素材归因、留存或利润数据。